# [SK 07 - AI Foundry Agents with Semantic Kernel vs. AI Foundry SDK's](https://github.com/microsoft/semantic-kernel/tree/main/python/samples/getting_started_with_agents/azure_ai_agent)
- How to use Azure AI Agents with Semantic Kernel.
- Dependencies: `pip install semantic-kernel[azure]`
- [Sample](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/getting_started_with_agents/azure_ai_agent/step1_azure_ai_agent.py)<br/><br/>

Note: it's worth to review the usage of AI Foundry Agents
- with [Python AI Foundry SDK](https://github.com/maurominella/aaas)
- with [C#](https://github.com/maurominella/aaas/tree/main/FoundryAgents06%20-%20AI%20Foundry%20Agent%20with%20BingGroundingTool)

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential

load_dotenv("./../config/credentials_my.env")

agent_name                  = "aiagent-PYTHON"
instructions                = "you are a clever agent"
user_inputs = [
    "Toggle the status of my second light.",
    "Toggle the status of the third light.",
    "Retrieve the status of all lights."
]

plugin_name                 = "Lights"

project_connection_string   = os.environ["PROJECT_CONNECTION_STRING"]
model_deployment_name       = os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT_NAME']

credential                  = DefaultAzureCredential()

# Native Plugin

In [2]:
# First, we define the plugin through its class...

class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
    
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# AI FOUNDRY PROJECT CLIENT

## AI Foundry SDK

In [3]:
from azure.ai.projects import AIProjectClient

aifoundry_project_client = AIProjectClient.from_connection_string(
    credential=credential, 
    conn_str=project_connection_string,
)

aifoundry_project_client

## Semantic Kernel SDK

In [4]:
from semantic_kernel.agents import AzureAIAgent

sk_project_client = AzureAIAgent.create_client(
    credential=credential,
    conn_str=project_connection_string,
)

sk_project_client

# AI FOUNDRY AGENT CREATION

## AI Foundry SDK
Single step:
- `create_agent` for agent **creation**

In [5]:
aifoundry_ai_agent = aifoundry_project_client.agents.create_agent(
    model=model_deployment_name,
    name=f"{agent_name}_aifoundry",
    instructions=instructions
)

aifoundry_ai_agent

{'id': 'asst_nWdBE1C9ymjrMAOsmkqB8518', 'object': 'assistant', 'created_at': 1743556071, 'name': 'aiagent-PYTHON_aifoundry', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {}, 'metadata': {}, 'response_format': 'auto'}

## Semantic Kernel SDK
Two steps:
- use `create_agent` for **agent definition**
- use `AzureAIAgent` for **client creation** (including **kernel**)

In [6]:
# use create_agent for agent definition

sk_ai_agent_definition = await sk_project_client.agents.create_agent(
    model=model_deployment_name,
    name=f"{agent_name}_SK",
    instructions=instructions
)
sk_ai_agent_definition

{'id': 'asst_uPj0PWaLZsY8ssNcpsVRAYHm', 'object': 'assistant', 'created_at': 1743556074, 'name': 'aiagent-PYTHON_SK', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {}, 'metadata': {}, 'response_format': 'auto'}

In [7]:
# use AzureAIAgent for client creation (including kernel)

sk_ai_agent = AzureAIAgent(
    client=sk_project_client,
    definition=sk_ai_agent_definition,
)

sk_ai_agent

AzureAIAgent(arguments=None, description=None, id='asst_uPj0PWaLZsY8ssNcpsVRAYHm', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000024E6EC04D10>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='aiagent-PYTHON_SK', prompt_template=None, client=<azure.ai.projects.aio._patch.AIProjectClient object at 0x0000024E6E9F01A0>, definition={'id': 'asst_uPj0PWaLZsY8ssNcpsVRAYHm', 'object': 'assistant', 'created_at': 1743556074, 'name': 'aiagent-PYTHON_SK', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {}, 'metadata': {}, 'response_format': 'auto'}, polling_options=RunPollingOptions(default_polling_interval=datetime.timedelta(microseconds=250000), default_polling_backoff=da

# ADD PLUGIN TO THE AGENT

## AI Foundry SDK

In [8]:
from azure.ai.projects.models import FunctionTool, ToolSet
from typing import Any, Callable, Set

# Instantiate the LightsPlugin class
lights_plugin = LightsPlugin()

# Create a set of the desired callable methods
plugin_functions: Set[Callable[..., Any]] = {
    lights_plugin.get_state,
    lights_plugin.change_state,
}

# Pass these functions to FunctionTool
functions = FunctionTool(plugin_functions)

# Add the plugin to the agent
aifoundry_project_client.agents.update_agent(tools=functions.definitions, agent_id=aifoundry_ai_agent.id)

aifoundry_ai_agent.items

<bound method _MyMutableMapping.items of {'id': 'asst_nWdBE1C9ymjrMAOsmkqB8518', 'object': 'assistant', 'created_at': 1743556071, 'name': 'aiagent-PYTHON_aifoundry', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {}, 'metadata': {}, 'response_format': 'auto'}>

## Semantic Kernel SDK

In [9]:
sk_ai_agent.kernel.add_plugin(
    plugin=LightsPlugin(),
    plugin_name=plugin_name,
)
sk_ai_agent

AzureAIAgent(arguments=None, description=None, id='asst_uPj0PWaLZsY8ssNcpsVRAYHm', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000024E6EC04D10>, plugins={'Lights': KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False,

# CREATE A NEW THREAD (EVEN AN EMPTY ONE)

## AI Foundry SDK

In [10]:
aifoundry_thread = aifoundry_project_client.agents.create_thread()
print(f"Created thread: {aifoundry_thread}\n")

Created thread: {'id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'object': 'thread', 'created_at': 1743556075, 'metadata': {}, 'tool_resources': {}}



## Semantic Kernel SDK
If no thread is provided, a new thread will be created and returned with the initial response

In [11]:
from semantic_kernel.agents import AzureAIAgentThread

thread: AzureAIAgentThread = None

# MESSAGE(S) CREATION

## AI Foundry SDK

In [12]:
for user_input in user_inputs:
    message = aifoundry_project_client.agents.create_message(
        thread_id=aifoundry_thread.id, 
        role="user", 
        content=user_input,
    )
    print(f"Created message: {message}")

Created message: {'id': 'msg_JoB7mDqdQq42nFWtIUxePTYG', 'object': 'thread.message', 'created_at': 1743556076, 'assistant_id': None, 'thread_id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'run_id': None, 'role': 'user', 'content': [{'type': 'text', 'text': {'value': 'Toggle the status of my second light.', 'annotations': []}}], 'attachments': [], 'metadata': {}}
Created message: {'id': 'msg_bPvkn5nkZqvFr8AiSdrwo3Ne', 'object': 'thread.message', 'created_at': 1743556077, 'assistant_id': None, 'thread_id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'run_id': None, 'role': 'user', 'content': [{'type': 'text', 'text': {'value': 'Toggle the status of the third light.', 'annotations': []}}], 'attachments': [], 'metadata': {}}
Created message: {'id': 'msg_vr9xwszoSOknItBLdBNJypJn', 'object': 'thread.message', 'created_at': 1743556077, 'assistant_id': None, 'thread_id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'run_id': None, 'role': 'user', 'content': [{'type': 'text', 'text': {'value': 'Retrieve the status of all

## Semantic Kernel SDK

In [13]:
# user messages do not need to be packaged into a message, since they can be passed as simple strings

# RUN THE AGENT

## AI Foundry SDK

In [14]:
%%time
import time

print(f"Running the agent {aifoundry_thread.id} on the thread {aifoundry_ai_agent.id}")

run = aifoundry_project_client.agents.create_run(thread_id=aifoundry_thread.id, agent_id=aifoundry_ai_agent.id)

while run.status in ['queued', 'in_progress', 'cancelling']:
    time.sleep(1)
    run = aifoundry_project_client.agents.get_run(thread_id=aifoundry_thread.id, run_id=run.id)
    print(f"Run status: {run.status}")

print(f"Run finished with status: {run.status}.\n\nRun: {run}")

if run.status == "failed":
    # Check if you got "Rate limit is exceeded.", then you want to get more quota
    print(f"Run failed: {run.last_error}")

Running the agent thread_w5CZamsF6Kyo1nlZIeCTCyvZ on the thread asst_nWdBE1C9ymjrMAOsmkqB8518
Run status: RunStatus.REQUIRES_ACTION
Run finished with status: RunStatus.REQUIRES_ACTION.

Run: {'id': 'run_sA7OiU9XAwlLURIXJQ9acQjA', 'object': 'thread.run', 'created_at': 1743556079, 'assistant_id': 'asst_nWdBE1C9ymjrMAOsmkqB8518', 'thread_id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'status': 'requires_action', 'started_at': 1743556079, 'expires_at': 1743556679, 'cancelled_at': None, 'failed_at': None, 'completed_at': None, 'required_action': {'type': 'submit_tool_outputs', 'submit_tool_outputs': {'tool_calls': [{'id': 'call_WyqgmuDAaV5SePFxStqUh1za', 'type': 'function', 'function': {'name': 'get_state', 'arguments': '{}'}}]}}, 'last_error': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [{'type': 'function', 'function': {'name': 'change_state', 'description': 'Changes the state of the light.', 'parameters': {'type': 'object', 'properties': {'id': {'type': 'integer'

In [ ]:
# TO BE FIXED FOR AI FOUNDRY

In [15]:
# KEEP RUNNING UNTIL RunStatus.COMPLETED

# just for checking: analyze the current status
import time, json
from azure.ai.projects.models import FunctionTool, ToolOutput

run = aifoundry_project_client.agents.get_run(thread_id=aifoundry_thread.id, run_id=run.id)
print(f"Initial run status: {run.status}")
print(f"\nRequired action(s): {run.required_action}")
print(f"\nWe need to run {len(run.required_action.submit_tool_outputs.tool_calls)} tool call(s): {run.required_action.submit_tool_outputs.tool_calls}")

i = 0
tool_outputs = []
for tool_call in run.required_action.submit_tool_outputs.tool_calls:
    i += 1
    output = functions.execute(tool_call)
    print(f"output: {output}")
    output=json.dumps(output) # TRYING TO PATCH, BUT IT STILL DOES NOT WORK
    
    print(f"\n{i} - Executing tool_call {tool_call.function.name} ({tool_call.id}) >>> output: {output}")
    tool_outputs.append(
        ToolOutput(
            tool_call_id=tool_call.id,
            output=output
        )
    )
    
run = aifoundry_project_client.agents.submit_tool_outputs_to_run(
    thread_id=aifoundry_thread.id, run_id=run.id, tool_outputs=tool_outputs
)

while run.status in ["queued", "in_progress"]:
    time.sleep(1)
    run = aifoundry_project_client.agents.get_run(thread_id=aifoundry_thread.id, run_id=run.id)
    print(f"\nFinal run status: {run.status}")

Initial run status: RunStatus.REQUIRES_ACTION

Required action(s): {'type': 'submit_tool_outputs', 'submit_tool_outputs': {'tool_calls': [{'id': 'call_WyqgmuDAaV5SePFxStqUh1za', 'type': 'function', 'function': {'name': 'get_state', 'arguments': '{}'}}]}}

We need to run 1 tool call(s): [{'id': 'call_WyqgmuDAaV5SePFxStqUh1za', 'type': 'function', 'function': {'name': 'get_state', 'arguments': '{}'}}]
output: [{'id': 0, 'name': 'Table Lamp', 'is_on': False}, {'id': 1, 'name': 'Porch light', 'is_on': False}, {'id': 2, 'name': 'Chandelier', 'is_on': False}]

1 - Executing tool_call get_state (call_WyqgmuDAaV5SePFxStqUh1za) >>> output: [{"id": 0, "name": "Table Lamp", "is_on": false}, {"id": 1, "name": "Porch light", "is_on": false}, {"id": 2, "name": "Chandelier", "is_on": false}]

Final run status: RunStatus.IN_PROGRESS

Final run status: RunStatus.COMPLETED


## Semantic Kernel SDK

In [16]:
for user_input in user_inputs:
    print(f"# User: {user_input}")
    # Invoke the agent with the specified message for response
    response = await sk_ai_agent.get_response(messages=user_input, thread=thread) # or sk_ai_agent.invoke(messages=user_input, thread=thread)
    thread = response.thread
    print(f"--> # {response.name}: {response}\n")

# User: Toggle the status of my second light.
--> # aiagent-PYTHON_SK: The status of your second light, the "Porch light," has been toggled on.

# User: Toggle the status of the third light.
--> # aiagent-PYTHON_SK: The status of your third light, the "Chandelier," has been toggled on.

# User: Retrieve the status of all lights.
--> # aiagent-PYTHON_SK: Here is the status of all your lights:

1. **Table Lamp**: Off
2. **Porch light**: On
3. **Chandelier**: On



# Fetch messages from the thread after the agent run execution

## AI Foundry SDK

In [17]:
from azure.ai.projects.models import MessageTextContent, MessageImageFileContent

if run.status == 'completed':    
    messages = aifoundry_project_client.agents.list_messages(thread_id=aifoundry_thread.id)
    messages_nr = len(messages.data)
    print(f"Here are the {messages_nr} messages:\n")
    
    for i, message in enumerate(reversed(messages.data), 1):
        j = 0
        print(f"\n===== MESSAGE {i} =====")
        for c in message.content:
            j +=1
            if (type(c) is MessageImageFileContent):
                print(f"\nCONTENT {j} (MessageImageFileContent) --> image_file id: {c.image_file.file_id}")
            elif (type(c) is MessageTextContent):
                print(f"\nCONTENT {j} (MessageTextContent) --> Text: {c.text.value}")
                for a in c.text.annotations:
                    print(f">>> Annotation in MessageTextContent {j} of message {i}: {a.text}\n")

else:
    print(f"Sorry, I can't proceed because the run status is {run.status}")

Here are the 4 messages:


===== MESSAGE 1 =====

CONTENT 1 (MessageTextContent) --> Text: Toggle the status of my second light.

===== MESSAGE 2 =====

CONTENT 1 (MessageTextContent) --> Text: Toggle the status of the third light.

===== MESSAGE 3 =====

CONTENT 1 (MessageTextContent) --> Text: Retrieve the status of all lights.

===== MESSAGE 4 =====

CONTENT 1 (MessageTextContent) --> Text: Here are the current statuses of all lights:

1. **Table Lamp**: Off
2. **Porch Light**: Off
3. **Chandelier**: Off

Would you like me to toggle the status for any specific light?


## Semantic Kernel SDK

In [18]:
# Get ready to print all messages of a AzureAIAgentThread object

async def print_messages(thread: AzureAIAgentThread):
    i=0
    messages = [message async for message in thread.get_messages()] # this doesn't work: messages = await thread.get_messages()
    for cmc in messages: # ChatMessageContent
        if cmc.inner_content is None:
            i += 1
            if cmc.role.value == "user" or cmc.role.value == "assistant":
                print(f"{i} - Role: {cmc.role.value}, text: {cmc.items[0].text}")
            elif cmc.role.value == "tool":
                print(f"{i} - Role: {cmc.role.value}, function_name: {cmc.items[0].name}")
        else:
            for choice in cmc.inner_content.choices:
                if choice.message.tool_calls is None:
                    i += 1
                    print(f"{i} - Finish reason: {choice.finish_reason}")
                else:
                    for tc in choice.message.tool_calls:
                        i += 1
                        print (f"{i} - Function call: {tc.function.name}({tc.function.arguments})")
    return messages

messages = await print_messages(thread)

1 - Role: assistant, text: Here is the status of all your lights:

1. **Table Lamp**: Off
2. **Porch light**: On
3. **Chandelier**: On
2 - Role: user, text: Retrieve the status of all lights.
3 - Role: assistant, text: The status of your third light, the "Chandelier," has been toggled on.
4 - Role: user, text: Toggle the status of the third light.
5 - Role: assistant, text: The status of your second light, the "Porch light," has been toggled on.
6 - Role: user, text: Toggle the status of my second light.


# HIC SUNT LEONES

# TEARDOWN

## AI Foundry SDK

In [19]:
# delete thread

print(f"Deleting thread {aifoundry_thread}...")
aifoundry_project_client.agents.delete_thread(aifoundry_thread.id)

Deleting thread {'id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'object': 'thread', 'created_at': 1743556075, 'metadata': {}, 'tool_resources': {}}...


{'id': 'thread_w5CZamsF6Kyo1nlZIeCTCyvZ', 'object': 'thread.deleted', 'deleted': True}

In [20]:
# delete agent
aifoundry_project_client.agents.delete_agent(aifoundry_ai_agent.id)

{'id': 'asst_nWdBE1C9ymjrMAOsmkqB8518', 'object': 'assistant.deleted', 'deleted': True}

# Semantic Kernel SDK

In [21]:
# delete thread

print(f"Deleting thread {thread.id}...")
await thread.delete()

Deleting thread thread_mjzkRZfgumSZHzI8PkuFX5Rx...


In [22]:
# delete agent
await sk_project_client.agents.delete_agent(sk_ai_agent.id)

{'id': 'asst_uPj0PWaLZsY8ssNcpsVRAYHm', 'object': 'assistant.deleted', 'deleted': True}

# HIC SUNT LEONES

In [24]:
# delete all agents
agents_list = aifoundry_project_client.agents.list_agents(limit=100).data # max limit is 100
print(f"There are {len(agents_list)} agents to delete")

i = 0
for agent in agents_list:
    i += 1
    print(f"Agent {i}/{len(agents_list)}: Agent {agent.name} ({agent.id})) is being deleted...")
    aifoundry_project_client.agents.delete_agent(agent.id) # comment / un-comment this line if you want to delete it

There are 0 agents to delete
